# Notebook 07 · Final evaluation (first run): metrics, equivalence test and test set

The analyses are those of the frozen statistical plan (registered on OSF; fingerprint `c3752d247a0173f1`):
nothing here selects a model, tunes a parameter or changes the decision rule.

**What it does:**

1. Checks that the 15 jobs of notebook 06 finished with the frozen code.
2. Uses each of the 15 models to predict the test set (CT-ICH) and the equivocal examinations of its fold.
3. Computes the primary comparison (RetinaNet versus YOLOv8m without mosaic) and the secondary and
   sensitivity analyses in cross-validation.
4. Computes the test-set result **once**: it is written to the Drive and, if the notebook is run again, it
   is read from the file instead of recomputed.
5. Measures prediction time on a CPU.

**Note:** in this run the RetinaNet arm had transposed boxes (deviation 1 of the plan). The evaluation with
the corrected RetinaNet is in notebook 09, which reuses the YOLOv8m predictions and test results of this run.

**Before running:** the 15 jobs finished (cell 4 of notebook 06 reporting 15 of 15); `src/ichdet`
**version 0.3.0** copied to `MyDrive/ich-benchmark/codigo/ichdet/`; L4 GPU.

**Time:** about 1 hour (CT-ICH prediction by the 15 models and 10,000 bootstrap resamples per analysis).

**Language:** documentation and comments are in English; identifiers, printed messages and the column
names of the files written here are in Portuguese, exactly as in the run that produced the results.

In [ ]:
# 1. Drive, libraries, GPU and check of the frozen code
import json, os, platform, socket, subprocess, sys, time, zipfile
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = Path('/content/drive/MyDrive/ich-benchmark')
    DISCO = Path('/content')                         # Colab disk: reading many images from the Drive is slow
    from importlib.metadata import version
    numpy_antes = version('numpy')
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'monai==1.5.2', 'ultralytics>=8.3,<8.4'],
                   check=True)
    if version('numpy') != numpy_antes:
        raise SystemExit(f'O pip trocou o numpy ({numpy_antes} -> {version("numpy")}). '
                         'Reinicie a sessão (Ambiente de execução → Reiniciar sessão) e rode de novo.')
except ImportError:                                  # local test: RAIZ mimics the Drive folder
    RAIZ = Path(os.environ['EXPERIMENTO_LOCAL'])
    DISCO = RAIZ / 'disco_local'

CODIGO = RAIZ / 'codigo'
sys.path.insert(0, str(CODIGO))
PREPARO = RAIZ / 'dados' / 'tc' / 'preparo'
SAIDA = RAIZ / 'dados' / 'tc' / 'experimento' / 'completo'
CQ500, CTICH = DISCO / 'cq500_512', DISCO / 'ctich_512'

import numpy as np, torch, monai, ultralytics
import ichdet
from ichdet import data_io, evaluate, experiment, final, training_config as cfg

CONGELADO = 'c3752d247a0173f1'                             # fingerprint of the code registered with the plan (protocol v1.1)
AMBIENTE = {
    'ichdet_version': ichdet.__version__, 'frozen_code_fingerprint': experiment.frozen_fingerprint(CODIGO / 'ichdet'),
    'python': sys.version.split()[0], 'numpy': np.__version__, 'torch': torch.__version__, 'monai': monai.__version__,
    'ultralytics': ultralytics.__version__,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
    'cpu': platform.processor() or platform.machine(), 'cpu_count': os.cpu_count(),
    'session': f'{socket.gethostname()}-{os.getpid()}', 'started': time.strftime('%Y-%m-%d %H:%M:%S'),
}
print(json.dumps(AMBIENTE, indent=2))
assert AMBIENTE['frozen_code_fingerprint'] == CONGELADO, (
    'O código no Drive não é o congelado. Copie de novo a pasta src/ichdet do repositório.')
assert torch.cuda.is_available() or 'EXPERIMENTO_LOCAL' in os.environ, 'Ative a GPU (Ambiente de execução → Alterar o tipo).'

In [ ]:
# 2. The 15 jobs, the images on the Colab disk and the reference boxes
trabalhos = {}
for k, braco in experiment.job_order():
    f = experiment.job_dir(SAIDA, k, braco) / 'job.json'
    assert f.exists(), f'rodada {k} / {braco}: treino não terminou. Termine o notebook 06 antes.'
    j = json.loads(f.read_text())
    assert j['epochs'] == cfg.EPOCHS and j['environment']['frozen_code_fingerprint'] == CONGELADO, f'rodada {k} / {braco}'
    assert final.weights_path(SAIDA, k, braco).exists(), f'rodada {k} / {braco}: pesos não encontrados'
    trabalhos[(k, braco)] = j
print(f'{len(trabalhos)} treinos prontos, todos com o código congelado e {cfg.EPOCHS} épocas')

for pasta, arquivo in ((CQ500, 'cq500_512.zip'), (CTICH, 'ctich_512.zip')):
    if not (pasta / 'index.csv').exists():
        inicio = time.time()
        with zipfile.ZipFile(PREPARO / arquivo) as z:
            z.extractall(pasta)
        print(f'{arquivo}: descompactado em {time.time() - inicio:.0f} s')

import csv
parte = {}                                           # examination -> fold (five folds, by patient)
with open(CQ500 / 'index.csv', newline='') as f:
    for r in csv.DictReader(f):
        parte[r['exame']] = int(r['parte'])

REF = {k: data_io.load_monai_split(CQ500 / 'monai' / f'fold{k}.json') for k in experiment.FOLDS}
duvidosos = json.loads((CQ500 / 'monai' / 'equivocal.json').read_text())['validation']
DUVIDOSOS = {k: [e for e in duvidosos if parte[e['exam']] == k] for k in experiment.FOLDS}
REF_DUV = {k: {data_io.image_key(e['image']): (e['exam'], np.zeros((0, 4))) for e in DUVIDOSOS[k]} for k in experiment.FOLDS}
teste = json.loads((CTICH / 'monai' / 'external.json').read_text())['validation']
REF_TESTE = data_io.load_monai_split(CTICH / 'monai' / 'external.json')

esperado = json.loads((PREPARO / 'export_512_summary.json').read_text())
for k in experiment.FOLDS:
    assert len(REF[k]) == esperado['cq500_folds_main_analysis'][f'fold{k}']['slices'], k
assert sum(len(v) for v in DUVIDOSOS.values()) == esperado['cq500_equivocal_slices']
assert len(teste) == esperado['ct_ich']['slices']
print(f"validação: {sum(len(v) for v in REF.values())} imagens; duvidosos: {len(duvidosos)}; teste (CT-ICH): {len(teste)}")

In [ ]:
# 3. Missing predictions: test set and equivocal examinations of each fold (GPU; resumes after a disconnection)
for (k, braco), j in trabalhos.items():
    d = experiment.job_dir(SAIDA, k, braco)
    for nome, entradas, raiz in (('pred_test.csv', teste, CTICH), ('pred_equivocal.csv', DUVIDOSOS[k], CQ500)):
        if (d / nome).exists():
            continue
        inicio = time.time()
        p = final.predict(braco, final.weights_path(SAIDA, k, braco), entradas, raiz)
        data_io.save_predictions(d / f'{nome}.tmp', p)
        (d / f'{nome}.tmp').replace(d / nome)
        print(f'rodada {k} / {braco}: {nome} em {time.time() - inicio:.0f} s')

def predicoes(k, braco, nome='pred_validation.csv'):
    return data_io.load_predictions(experiment.job_dir(SAIDA, k, braco) / nome)

PRED = {(k, b): predicoes(k, b) for k, b in trabalhos}
PRED_DUV = {(k, b): predicoes(k, b, 'pred_equivocal.csv') for k, b in trabalhos}
PRED_TESTE = {(k, b): predicoes(k, b, 'pred_test.csv') for k, b in trabalhos}
print('predições carregadas')

In [ ]:
# 4. Cross-validation: primary comparison, effect of mosaic, sensitivity analyses and per-subtype analysis
A, B, C = 'retinanet', 'yolo_no_mosaic', 'yolo_mosaic'
K = experiment.FOLDS

def rodadas(x, y, ref=REF, pred=PRED):
    return [(ref[k], pred[(k, x)], pred[(k, y)]) for k in K]

validacao = {'primary_A_vs_B': evaluate.compare_crossval(rodadas(A, B))}
validacao['mosaic_C_vs_B'] = evaluate.compare_crossval(rodadas(C, B))
validacao['mosaic_A_vs_C'] = evaluate.compare_crossval(rodadas(A, C))

# Sensitivity analysis 1: predictions of the five folds pooled in a single table
validacao['sensitivity_pooled'] = final.compare_pooled(rodadas(A, B))
# Sensitivity analysis 2: equivocal examinations added to validation as negatives
com_duv = {}
for braco in (A, B):
    for k in K:
        com_duv[(k, braco)] = final.merge((REF[k], PRED[(k, braco)]), (REF_DUV[k], PRED_DUV[(k, braco)]))
validacao['sensitivity_equivocal_as_negative'] = evaluate.compare_crossval(
    [(com_duv[(k, A)][0], com_duv[(k, A)][1], com_duv[(k, B)][1]) for k in K])
# Sensitivity analysis 3: without examinations with BHX boxes but negative for the majority of CQ500 readers
discordantes = ['CQ500-CT-0', 'CQ500-CT-134', 'CQ500-CT-176', 'CQ500-CT-187', 'CQ500-CT-224', 'CQ500-CT-234', 'CQ500-CT-24', 'CQ500-CT-257', 'CQ500-CT-283', 'CQ500-CT-284', 'CQ500-CT-288', 'CQ500-CT-291', 'CQ500-CT-31', 'CQ500-CT-337', 'CQ500-CT-340', 'CQ500-CT-345', 'CQ500-CT-351', 'CQ500-CT-374', 'CQ500-CT-380', 'CQ500-CT-391', 'CQ500-CT-398', 'CQ500-CT-414', 'CQ500-CT-483', 'CQ500-CT-486', 'CQ500-CT-56', 'CQ500-CT-63', 'CQ500-CT-67']                          # from cq500_series_escolhidas.csv (votos_ich <= 1, i.e. at most one positive read)
assert set(discordantes) <= set(parte)
sem = {k: final.subset(REF[k], PRED[(k, A)], discordantes) for k in K}
validacao['sensitivity_without_bhx_cq500_disagreement'] = evaluate.compare_crossval(
    [(sem[k][0], sem[k][1], final.subset(REF[k], PRED[(k, B)], discordantes)[1]) for k in K])
validacao['sensitivity_without_bhx_cq500_disagreement']['exams_removed'] = discordantes

# Sensitivity per haemorrhage subtype at 1 false positive per examination (exploratory)
tipos = {}
for k in K:
    tipos.update(final.box_subtypes(REF[k], CQ500 / 'boxes_512.csv'))
validacao['subtype_sensitivity_1fp'] = {b: final.subtype_sensitivity([(REF[k], PRED[(k, b)]) for k in K], tipos)
                                        for b in (A, B, C)}

(SAIDA / 'results_validation.json').write_text(json.dumps(validacao, indent=2, default=float))
p = validacao['primary_A_vs_B']['equivalence_A_minus_B']
print(f"AP@0.5 RetinaNet {validacao['primary_A_vs_B']['A']['ap50']:.3f}; YOLO sem mosaico {validacao['primary_A_vs_B']['B']['ap50']:.3f}")
print(f"Diferença {p['difference']:+.3f}; IC 90% [{p['ci90'][0]:+.3f}, {p['ci90'][1]:+.3f}]; margem ±{p['margin']}")
print('Leitura pré-especificada:', final.reading(p['ci90'], p['margin']))

In [ ]:
# 5. Test set (CT-ICH): computed ONCE; afterwards only read from the file
arquivo_teste = SAIDA / 'results_test.json'
if arquivo_teste.exists():
    resultado_teste = json.loads(arquivo_teste.read_text())
    print(f"resultado da base de teste lido do arquivo (calculado em {resultado_teste['computed']})")
else:
    modelos = {b: [PRED_TESTE[(k, b)] for k in K] for b in (A, B, C)}
    resultado_teste = {
        'computed': time.strftime('%Y-%m-%d %H:%M:%S'),
        'A_vs_B': evaluate.compare_test(REF_TESTE, modelos[A], modelos[B]),
        'C_vs_B': evaluate.compare_test(REF_TESTE, modelos[C], modelos[B]),
        'A_vs_C': evaluate.compare_test(REF_TESTE, modelos[A], modelos[C]),
        'generalisation_drop': {b: final.generalisation_drop([(REF[k], PRED[(k, b)]) for k in K], REF_TESTE, modelos[b])
                                for b in (A, B, C)},
    }
    arquivo_teste.write_text(json.dumps(resultado_teste, indent=2, default=float))
t = resultado_teste['A_vs_B']
print(f"Teste: AP@0.5 RetinaNet {t['A']['ap50']:.3f}; YOLO sem mosaico {t['B']['ap50']:.3f}; "
      f"diferença {t['equivalence_A_minus_B']['difference']:+.3f}")
for b, q in resultado_teste['generalisation_drop'].items():
    print(f"{b:15s} validação {q['validation_ap50']:.3f} → teste {q['test_ap50']:.3f} (queda {q['drop']:+.3f})")

In [ ]:
# 6. CPU prediction time (fold-1 model of each arm, 100 CT-ICH images, one at a time)
latencia = {b: final.cpu_latency(b, final.weights_path(SAIDA, 1, b), teste, CTICH) for b in (A, B, C)}
for b, q in latencia.items():
    print(f"{b:15s} mediana {q['median_seconds_per_slice'] * 1000:.0f} ms por imagem ({q['torch_threads']} threads)")

In [ ]:
# 7. Single record of the results
resumo = {'environment': AMBIENTE, 'jobs': len(trabalhos), 'validation': validacao, 'test': resultado_teste,
          'cpu_latency': latencia}
(SAIDA / 'final_results.json').write_text(json.dumps(resumo, indent=2, default=float))
print('final_results.json gravado em', SAIDA)

## After running

1. `final_results.json`, `results_validation.json` and `results_test.json` in `dados/tc/experimento/completo/`
   on the Drive hold the results of the first run (included in this repository under `data/`).
2. **Do not run cell 5 again with different code:** the test-set result is the one computed the first time.